# ASR Robustness Evaluation for Chinese Opera Singing

This notebook provides a comprehensive framework for evaluating ASR models on Chinese opera singing data, comparing performance between original and audio-modified versions.

## Features
- **Multi-model evaluation**: Whisper, wav2vec2, and Vosk
- **Audio processing**: Shorten elongated voiced segments
- **Detailed error analysis**: Substitution, insertion, deletion rates
- **Remote server ready**: Upload data and run evaluations

## Setup Instructions
1. Upload your data files using the Data Upload section
2. Configure paths and parameters
3. Run the evaluation pipeline

---

# Install required packages (uncomment if needed)
# !pip install torch torchaudio transformers librosa soundfile jiwer opencc-python-reimplemented vosk

import os
import sys
import csv
import json
import warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import time

# Add scripts to path
sys.path.append('scripts')

# Suppress warnings
warnings.filterwarnings('ignore')

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Import custom modules
from utils.audio_utils import load_audio, check_audio_file
from utils.text_utils import normalize_text
from models.asr_models import get_model
from evaluation.evaluate_models import evaluate_dataset, save_evaluation_summary, print_results_table
from audio_processing.voice_segmentation import process_elongated_segments
from utils.timing_utils import initialize_timing, log_cell, log_notebook_completion

# Initialize timing system
notebook_start_time = time.time()
timer = initialize_timing(f"results/execution_logs/timing_log_{CONFIG['run_id']}.json" if 'CONFIG' in globals() else None)

print("✅ Environment setup complete")

In [ ]:
# Configuration - Update these paths based on your uploaded data
import pandas as pd
from datetime import datetime

# Generate unique run ID
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"run_{RUN_ID}"

# Count dataset size for optimal configuration
dataset_size = 0
if os.path.exists("data/test.csv"):
    dataset_size = len(pd.read_csv("data/test.csv"))

# Get optimal parallel configuration
from scripts.evaluation.parallel_evaluate_models import get_optimal_config
optimal_config = get_optimal_config(dataset_size, DEVICE)

CONFIG = {
    # Data paths
    "original_csv": "data/test.csv",
    "output_audio_dir": f"data/clips/test_shortened_{RUN_ID}",
    "shortened_csv": f"data/test_shortened_{RUN_ID}.csv",
    
    # Model paths
    "vosk_model_path": "models/vosk-model-small-cn-0.22",  # Set to None if not using Vosk
    
    # Output paths with timestamps for multiple runs
    "results_dir": "results",
    "predictions_dir": f"results/predictions/{RUN_NAME}",
    "comparisons_dir": f"results/comparisons/{RUN_NAME}",
    "multi_run_dir": "results/multi_run_comparisons",
    
    # Audio processing parameters
    "target_sr": 16000,
    "min_elongated_sec": 0.50,
    "shorten_factor": 0.75,
    
    # Evaluation settings
    "models_to_eval": ["whisper", "wav2vec2", "vosk"],  # Remove "vosk" if not available
    "verbose": True,
    
    # Parallel processing settings
    "parallel_enabled": True,  # Enable/disable parallel processing
    "max_workers": optimal_config["max_workers"],  # Number of parallel threads
    "batch_size": optimal_config["batch_size"],  # Samples per batch
    
    # Run metadata
    "run_id": RUN_ID,
    "run_name": RUN_NAME,
    "run_timestamp": datetime.now().isoformat()
}

# Create output directories
for dir_path in [CONFIG["results_dir"], CONFIG["predictions_dir"], CONFIG["comparisons_dir"], CONFIG["output_audio_dir"], CONFIG["multi_run_dir"]]:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

print(f"\n🆔 Run ID: {RUN_ID}")
print(f"📁 This run's outputs will be saved in: {CONFIG['predictions_dir']}")
print(f"📊 Comparisons will be saved in: {CONFIG['comparisons_dir']}")
print(f"🚀 Parallel processing: {CONFIG['parallel_enabled']}")
print(f"⚡ Optimal config: {CONFIG['max_workers']} workers, {CONFIG['batch_size']} batch size")

def process_shortened_audio():
    """Process original audio to create shortened version"""
    with log_cell("Audio Processing - Shortening Elongated Segments"):
        if not os.path.exists(CONFIG["original_csv"]):
            print(f"❌ Original CSV not found: {CONFIG['original_csv']}")
            return False
        
        print("🔄 Processing audio files to shorten elongated segments...")
        
        # Load original dataset
        with open(CONFIG["original_csv"], "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        
        rows_out = []
        summary = []
        
        for i, row in enumerate(tqdm(rows, desc="Processing audio")):
            input_audio = row["audio"].strip()
            text = row["text"].strip()
            
            # Generate output path
            out_name = Path(input_audio).name
            output_audio = os.path.join(CONFIG["output_audio_dir"], out_name)
            
            # Skip if already processed
            if os.path.exists(output_audio):
                # Load existing info if possible
                rows_out.append({
                    "audio": output_audio,
                    "text": text,
                })
                continue
            
            # Process audio
            try:
                info = process_elongated_segments(
                    input_audio, 
                    output_audio,
                    target_sr=CONFIG["target_sr"],
                    min_elongated_sec=CONFIG["min_elongated_sec"],
                    shorten_factor=CONFIG["shorten_factor"]
                )
                
                rows_out.append({
                    "audio": output_audio,
                    "text": text,
                })
                summary.append(info)
                
                if CONFIG["verbose"]:
                    print(f"  Processed {out_name}: {info['original_duration_sec']:.2f}s -> {info['new_duration_sec']:.2f}s")
                    
            except Exception as e:
                print(f"  ❌ Error processing {input_audio}: {e}")
                # Use original audio as fallback
                rows_out.append({
                    "audio": input_audio,
                    "text": text,
                })
        
        # Save shortened CSV
        with open(CONFIG["shortened_csv"], "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["audio", "text"])
            writer.writeheader()
            writer.writerows(rows_out)
        
        print(f"✅ Saved shortened CSV: {CONFIG['shortened_csv']}")
        
        # Print summary
        if summary:
            total_original = sum(s['original_duration_sec'] for s in summary)
            total_new = sum(s['new_duration_sec'] for s in summary)
            total_modified = sum(s['modified_segments'] for s in summary)
            
            print(f"\n📊 Processing Summary:")
            print(f"  Files processed: {len(summary)}")
            print(f"  Total original duration: {total_original:.2f}s")
            print(f"  Total new duration: {total_new:.2f}s")
            print(f"  Duration reduction: {(1 - total_new/total_original)*100:.1f}%")
            print(f"  Total modified segments: {total_modified}")
        
        return True

# Run audio processing
process_shortened_audio()

def evaluate_original_data():
    """Evaluate ASR models on original audio data"""
    with log_cell("Model Evaluation - Original Data"):
        if not os.path.exists(CONFIG["original_csv"]):
            print(f"❌ Original CSV not found: {CONFIG['original_csv']}")
            return []
        
        results = []
        
        for model_name in CONFIG["models_to_eval"]:
            print(f"\n🎯 Evaluating {model_name} on original data...")
            
            output_pred_csv = os.path.join(CONFIG["predictions_dir"], f"{model_name}_original_auto.csv")
            
            try:
                if CONFIG.get("parallel_enabled", False):
                    # Use parallel evaluation
                    from scripts.evaluation.parallel_evaluate_models import evaluate_dataset_parallel
                    
                    result = evaluate_dataset_parallel(
                        model_name=model_name,
                        input_csv=CONFIG["original_csv"],
                        output_pred_csv=output_pred_csv,
                        device=DEVICE,
                        vosk_model_path=CONFIG["vosk_model_path"],
                        max_workers=CONFIG["max_workers"],
                        batch_size=CONFIG["batch_size"],
                        verbose=CONFIG["verbose"]
                    )
                else:
                    # Use sequential evaluation
                    result = evaluate_dataset(
                        model_name=model_name,
                        input_csv=CONFIG["original_csv"],
                        output_pred_csv=output_pred_csv,
                        device=DEVICE,
                        vosk_model_path=CONFIG["vosk_model_path"],
                        verbose=CONFIG["verbose"]
                    )
                
                result["condition"] = "original"
                result["run_id"] = CONFIG["run_id"]
                result["run_name"] = CONFIG["run_name"]
                result["timestamp"] = CONFIG["run_timestamp"]
                results.append(result)
                
            except Exception as e:
                print(f"❌ Error evaluating {model_name}: {e}")
        
        return results

# Run evaluation on original data
original_results = evaluate_original_data()

In [ ]:
def evaluate_shortened_data():
    """Evaluate ASR models on shortened audio data"""
    with log_cell("Model Evaluation - Shortened Data"):
        if not os.path.exists(CONFIG["shortened_csv"]):
            print(f"❌ Shortened CSV not found: {CONFIG['shortened_csv']}")
            return []
        
        results = []
        
        for model_name in CONFIG["models_to_eval"]:
            print(f"\n🎯 Evaluating {model_name} on shortened data...")
            
            output_pred_csv = os.path.join(CONFIG["predictions_dir"], f"{model_name}_shortened_auto.csv")
            
            try:
                if CONFIG.get("parallel_enabled", False):
                    # Use parallel evaluation
                    from scripts.evaluation.parallel_evaluate_models import evaluate_dataset_parallel
                    
                    result = evaluate_dataset_parallel(
                        model_name=model_name,
                        input_csv=CONFIG["shortened_csv"],
                        output_pred_csv=output_pred_csv,
                        device=DEVICE,
                        vosk_model_path=CONFIG["vosk_model_path"],
                        max_workers=CONFIG["max_workers"],
                        batch_size=CONFIG["batch_size"],
                        verbose=CONFIG["verbose"]
                    )
                else:
                    # Use sequential evaluation
                    result = evaluate_dataset(
                        model_name=model_name,
                        input_csv=CONFIG["shortened_csv"],
                        output_pred_csv=output_pred_csv,
                        device=DEVICE,
                        vosk_model_path=CONFIG["vosk_model_path"],
                        verbose=CONFIG["verbose"]
                    )
                
                result["condition"] = "shortened"
                result["run_id"] = CONFIG["run_id"]
                result["run_name"] = CONFIG["run_name"]
                result["timestamp"] = CONFIG["run_timestamp"]
                results.append(result)
                
            except Exception as e:
                print(f"❌ Error evaluating {model_name}: {e}")
        
        return results

# Run evaluation on shortened data
shortened_results = evaluate_shortened_data()

# Combine all results and analyze
with log_cell("Results Analysis and Visualization"):
    all_results = original_results + shortened_results

    if all_results:
        # Print results table
        print_results_table(all_results)
        
        # Save detailed results
        summary_json = os.path.join(CONFIG["comparisons_dir"], f"asr_compare_{CONFIG['run_id']}.json")
        summary_csv = os.path.join(CONFIG["comparisons_dir"], f"asr_compare_{CONFIG['run_id']}.csv")
        
        save_evaluation_summary(all_results, summary_json, summary_csv)
        
        print(f"\n📁 Results saved to:")
        print(f"  JSON: {summary_json}")
        print(f"  CSV: {summary_csv}")
        
        # Save run metadata
        run_metadata = {
            "run_id": CONFIG["run_id"],
            "run_name": CONFIG["run_name"],
            "timestamp": CONFIG["run_timestamp"],
            "config": CONFIG,
            "results_summary": all_results
        }
        
        metadata_path = os.path.join(CONFIG["comparisons_dir"], f"run_metadata_{CONFIG['run_id']}.json")
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(run_metadata, f, ensure_ascii=False, indent=2)
        
        print(f"  Metadata: {metadata_path}")
    else:
        print("❌ No evaluation results to analyze.")

In [ ]:
def create_results_package():
    """Create a comprehensive results package for download"""
    with log_cell("Results Package Creation"):
        if not all_results:
            print("❌ No results to package")
            return
        
        # Create comprehensive summary
        summary = {
            "run_id": CONFIG["run_id"],
            "run_name": CONFIG["run_name"],
            "timestamp": CONFIG["run_timestamp"],
            "config": CONFIG,
            "device": DEVICE,
            "total_samples": len(pd.read_csv(CONFIG["original_csv"])) if os.path.exists(CONFIG["original_csv"]) else 0,
            "models_evaluated": CONFIG["models_to_eval"],
            "results_summary": all_results
        }
        
        # Save comprehensive summary
        comprehensive_path = os.path.join(CONFIG["comparisons_dir"], f"comprehensive_summary_{CONFIG['run_id']}.json")
        with open(comprehensive_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)
        
        print(f"\n📦 Results Package Created:")
        print(f"  📄 Comprehensive Summary: {comprehensive_path}")
        print(f"  📊 Predictions Directory: {CONFIG['predictions_dir']}")
        print(f"  📈 Comparisons Directory: {CONFIG['comparisons_dir']}")
        
        # Create file list for easy download
        file_list = []
        for root, dirs, files in os.walk(CONFIG["predictions_dir"]):
            for file in files:
                file_list.append(os.path.join(root, file))
        for root, dirs, files in os.walk(CONFIG["comparisons_dir"]):
            for file in files:
                file_list.append(os.path.join(root, file))
        
        file_list_path = os.path.join(CONFIG["comparisons_dir"], f"file_list_{CONFIG['run_id']}.txt")
        with open(file_list_path, 'w') as f:
            for file_path in sorted(file_list):
                f.write(f"{file_path}\n")
        
        print(f"  📋 File List: {file_list_path}")
        print(f"\n✨ Total files generated: {len(file_list)}")
        
        return len(file_list)

# Create results package
total_files = create_results_package()

# Notebook completion and final logging
with log_cell("Final Completion Logging"):
    total_execution_time = time.time() - notebook_start_time
    
    # Create completion summary
    completion_summary = {
        "total_files_generated": total_files if 'total_files' in locals() else 0,
        "models_evaluated": CONFIG.get("models_to_eval", []),
        "runs_completed": len(all_results) if 'all_results' in locals() else 0,
        "output_directory": CONFIG.get("predictions_dir", "N/A")
    }
    
    # Log notebook completion
    log_notebook_completion(
        notebook_name="ASR Robustness Evaluation",
        total_time=total_execution_time,
        results_summary=completion_summary,
        log_file=f"results/execution_logs/timing_log_{CONFIG['run_id']}.json"
    )
    
    # Print execution summary
    print("\n" + "="*60)
    print("🎉 ASR ROBUSTNESS EVALUATION COMPLETED")
    print("="*60)
    
    minutes, seconds = divmod(total_execution_time, 60)
    print(f"🕐 Total execution time: {minutes:.0f}m {seconds:.1f}s")
    print(f"📁 Results saved in: {CONFIG.get('predictions_dir', 'N/A')}")
    print(f"📊 Comparisons saved in: {CONFIG.get('comparisons_dir', 'N/A')}")
    print(f"📝 Timing log: results/execution_logs/timing_log_{CONFIG['run_id']}.json")
    
    if 'all_results' in locals() and all_results:
        print(f"🎯 Models evaluated: {CONFIG.get('models_to_eval', [])}")
        print(f"📈 Total evaluations: {len(all_results)}")
        
        # Show best performing result
        best_result = min(all_results, key=lambda x: x['cer'])
        print(f"🏆 Best performance: {best_result['model']} ({best_result['condition']}) - CER: {best_result['cer']:.4f}")
    
    print("\n✨ Ready to download results and run comparisons!")
    print("="*60)

This section processes the original audio files to shorten elongated voiced segments, creating a modified version for robustness testing.

In [ ]:
def process_shortened_audio():
    """Process original audio to create shortened version"""
    if not os.path.exists(CONFIG["original_csv"]):
        print(f"❌ Original CSV not found: {CONFIG['original_csv']}")
        return False
    
    print("🔄 Processing audio files to shorten elongated segments...")
    
    # Load original dataset
    with open(CONFIG["original_csv"], "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    rows_out = []
    summary = []
    
    for i, row in enumerate(tqdm(rows, desc="Processing audio")):
        input_audio = row["audio"].strip()
        text = row["text"].strip()
        
        # Generate output path
        out_name = Path(input_audio).name
        output_audio = os.path.join(CONFIG["output_audio_dir"], out_name)
        
        # Skip if already processed
        if os.path.exists(output_audio):
            # Load existing info if possible
            rows_out.append({
                "audio": output_audio,
                "text": text,
            })
            continue
        
        # Process audio
        try:
            info = process_elongated_segments(
                input_audio, 
                output_audio,
                target_sr=CONFIG["target_sr"],
                min_elongated_sec=CONFIG["min_elongated_sec"],
                shorten_factor=CONFIG["shorten_factor"]
            )
            
            rows_out.append({
                "audio": output_audio,
                "text": text,
            })
            summary.append(info)
            
            if CONFIG["verbose"]:
                print(f"  Processed {out_name}: {info['original_duration_sec']:.2f}s -> {info['new_duration_sec']:.2f}s")
                
        except Exception as e:
            print(f"  ❌ Error processing {input_audio}: {e}")
            # Use original audio as fallback
            rows_out.append({
                "audio": input_audio,
                "text": text,
            })
    
    # Save shortened CSV
    with open(CONFIG["shortened_csv"], "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["audio", "text"])
        writer.writeheader()
        writer.writerows(rows_out)
    
    print(f"✅ Saved shortened CSV: {CONFIG['shortened_csv']}")
    
    # Print summary
    if summary:
        total_original = sum(s['original_duration_sec'] for s in summary)
        total_new = sum(s['new_duration_sec'] for s in summary)
        total_modified = sum(s['modified_segments'] for s in summary)
        
        print(f"\n📊 Processing Summary:")
        print(f"  Files processed: {len(summary)}")
        print(f"  Total original duration: {total_original:.2f}s")
        print(f"  Total new duration: {total_new:.2f}s")
        print(f"  Duration reduction: {(1 - total_new/total_original)*100:.1f}%")
        print(f"  Total modified segments: {total_modified}")
    
    return True

# Run audio processing
process_shortened_audio()

## 4. ASR Model Evaluation

### 4.1 Evaluate on Original Data

In [ ]:
def evaluate_original_data():
    """Evaluate ASR models on original audio data"""
    if not os.path.exists(CONFIG["original_csv"]):
        print(f"❌ Original CSV not found: {CONFIG['original_csv']}")
        return []
    
    results = []
    
    for model_name in CONFIG["models_to_eval"]:
        print(f"\n🎯 Evaluating {model_name} on original data...")
        
        output_pred_csv = os.path.join(CONFIG["predictions_dir"], f"{model_name}_original_auto.csv")
        
        try:
            result = evaluate_dataset(
                model_name=model_name,
                input_csv=CONFIG["original_csv"],
                output_pred_csv=output_pred_csv,
                device=DEVICE,
                vosk_model_path=CONFIG["vosk_model_path"],
                verbose=CONFIG["verbose"]
            )
            result["condition"] = "original"
            results.append(result)
            
        except Exception as e:
            print(f"❌ Error evaluating {model_name}: {e}")
    
    return results

# Run evaluation on original data
original_results = evaluate_original_data()

### 4.2 Evaluate on Shortened Data

In [ ]:
def evaluate_shortened_data():
    """Evaluate ASR models on shortened audio data"""
    if not os.path.exists(CONFIG["shortened_csv"]):
        print(f"❌ Shortened CSV not found: {CONFIG['shortened_csv']}")
        return []
    
    results = []
    
    for model_name in CONFIG["models_to_eval"]:
        print(f"\n🎯 Evaluating {model_name} on shortened data...")
        
        output_pred_csv = os.path.join(CONFIG["predictions_dir"], f"{model_name}_shortened_auto.csv")
        
        try:
            result = evaluate_dataset(
                model_name=model_name,
                input_csv=CONFIG["shortened_csv"],
                output_pred_csv=output_pred_csv,
                device=DEVICE,
                vosk_model_path=CONFIG["vosk_model_path"],
                verbose=CONFIG["verbose"]
            )
            result["condition"] = "shortened"
            results.append(result)
            
        except Exception as e:
            print(f"❌ Error evaluating {model_name}: {e}")
    
    return results

# Run evaluation on shortened data
shortened_results = evaluate_shortened_data()

## 5. Results Analysis and Visualization

In [ ]:
# Combine all results
all_results = original_results + shortened_results

if all_results:
    # Print results table
    print_results_table(all_results)
    
    # Save detailed results
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    summary_json = os.path.join(CONFIG["comparisons_dir"], f"asr_compare_{timestamp}.json")
    summary_csv = os.path.join(CONFIG["comparisons_dir"], f"asr_compare_{timestamp}.csv")
    
    save_evaluation_summary(all_results, summary_json, summary_csv)
    
    print(f"\n📁 Results saved to:")
    print(f"  JSON: {summary_json}")
    print(f"  CSV: {summary_csv}")
else:
    print("❌ No evaluation results to analyze.")

### 5.1 Visualization

In [ ]:
if all_results:
    # Convert to DataFrame for easier plotting
    results_df = pd.DataFrame(all_results)
    
    # Set up plotting style
    plt.style.use('seaborn-v0_8')
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('ASR Model Performance Comparison: Original vs Shortened Audio', fontsize=16)
    
    # 1. CER Comparison
    ax1 = axes[0, 0]
    pivot_cer = results_df.pivot(index='model', columns='condition', values='cer')
    pivot_cer.plot(kind='bar', ax=ax1)
    ax1.set_title('Character Error Rate (CER)')
    ax1.set_ylabel('CER')
    ax1.legend(title='Condition')
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Error Types Comparison
    ax2 = axes[0, 1]
    error_types = ['substitution_rate', 'insertion_rate', 'deletion_rate']
    for error_type in error_types:
        pivot_error = results_df.pivot(index='model', columns='condition', values=error_type)
        pivot_error.plot(kind='bar', ax=ax2, alpha=0.7, label=error_type.replace('_rate', ''))
    ax2.set_title('Error Type Rates')
    ax2.set_ylabel('Rate')
    ax2.legend()
    ax2.tick_params(axis='x', rotation=45)
    
    # 3. CER Improvement
    ax3 = axes[1, 0]
    pivot_improvement = pivot_cer.copy()
    if 'original' in pivot_improvement.columns and 'shortened' in pivot_improvement.columns:
        improvement = ((pivot_improvement['original'] - pivot_improvement['shortened']) / pivot_improvement['original'] * 100)
        improvement.plot(kind='bar', ax=ax3, color='green' if improvement.mean() > 0 else 'red')
        ax3.set_title('CER Improvement with Shortened Audio (%)')
        ax3.set_ylabel('Improvement (%)')
        ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax3.tick_params(axis='x', rotation=45)
    
    # 4. Model Performance Summary
    ax4 = axes[1, 1]
    model_summary = results_df.groupby('model')['cer'].mean().sort_values()
    model_summary.plot(kind='barh', ax=ax4)
    ax4.set_title('Average CER by Model')
    ax4.set_xlabel('Average CER')
    
    plt.tight_layout()
    plt.show()
    
    # Save plots
    plot_path = os.path.join(CONFIG["comparisons_dir"], f"asr_comparison_plots_{timestamp}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"📊 Plots saved to: {plot_path}")

### 5.2 Detailed Error Analysis

In [ ]:
if all_results:
    # Create detailed analysis table
    analysis_df = results_df.copy()
    
    # Calculate robustness metrics
    robustness_metrics = []
    
    for model in analysis_df['model'].unique():
        model_data = analysis_df[analysis_df['model'] == model]
        original_data = model_data[model_data['condition'] == 'original']
        shortened_data = model_data[model_data['condition'] == 'shortened']
        
        if len(original_data) > 0 and len(shortened_data) > 0:
            orig_cer = original_data['cer'].iloc[0]
            short_cer = shortened_data['cer'].iloc[0]
            
            robustness_metrics.append({
                'model': model,
                'original_cer': orig_cer,
                'shortened_cer': short_cer,
                'cer_change': short_cer - orig_cer,
                'cer_change_pct': ((short_cer - orig_cer) / orig_cer * 100),
                'robustness_score': abs(short_cer - orig_cer) / orig_cer  # Lower is more robust
            })
    
    robustness_df = pd.DataFrame(robustness_metrics)
    robustness_df = robustness_df.sort_values('robustness_score')
    
    print("\n🔍 Robustness Analysis (Lower robustness score = more robust to audio changes):")
    print(robustness_df.to_string(index=False, float_format='%.4f'))
    
    # Save detailed analysis
    analysis_path = os.path.join(CONFIG["comparisons_dir"], f"robustness_analysis_{timestamp}.csv")
    robustness_df.to_csv(analysis_path, index=False)
    print(f"\n📁 Detailed analysis saved to: {analysis_path}")

## 6. Export Results

In [ ]:
def create_results_package():
    """Create a comprehensive results package for download"""
    if not all_results:
        print("❌ No results to package")
        return
    
    # Create results summary
    summary = {
        "timestamp": timestamp,
        "config": CONFIG,
        "device": DEVICE,
        "total_samples": len(pd.read_csv(CONFIG["original_csv"])) if os.path.exists(CONFIG["original_csv"]) else 0,
        "models_evaluated": CONFIG["models_to_eval"],
        "results_summary": all_results
    }
    
    # Save comprehensive summary
    comprehensive_path = os.path.join(CONFIG["comparisons_dir"], f"comprehensive_summary_{timestamp}.json")
    with open(comprehensive_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    
    print(f"\n📦 Results Package Created:")
    print(f"  📄 Comprehensive Summary: {comprehensive_path}")
    print(f"  📊 Predictions Directory: {CONFIG['predictions_dir']}")
    print(f"  📈 Comparisons Directory: {CONFIG['comparisons_dir']}")
    
    # Create file list for easy download
    file_list = []
    for root, dirs, files in os.walk(CONFIG["results_dir"]):
        for file in files:
            file_list.append(os.path.join(root, file))
    
    file_list_path = os.path.join(CONFIG["results_dir"], "file_list.txt")
    with open(file_list_path, 'w') as f:
        for file_path in sorted(file_list):
            f.write(f"{file_path}\n")
    
    print(f"  📋 File List: {file_list_path}")
    print(f"\n✨ Total files generated: {len(file_list)}")

# Create results package
create_results_package()

## 7. Quick Start Summary

### For Remote Server Usage:

1. **Upload Data**: Use the file upload feature to upload:
   - Audio files to `data/clips/test_fixed/`
   - CSV file to `data/test.csv`
   - Vosk model (optional) to `models/vosk-model-small-cn-0.22/`

2. **Configure**: Update the CONFIG section with your file paths

3. **Run**: Execute cells in order:
   - Cell 1: Environment setup
   - Cell 2: Configuration and data verification
   - Cell 3: Audio processing (creates shortened versions)
   - Cell 4: Model evaluation (original and shortened)
   - Cell 5: Results analysis and visualization
   - Cell 6: Export results

### Key Outputs:
- **Predictions**: `results/predictions/` - Detailed predictions for each model/condition
- **Comparisons**: `results/comparisons/` - Summary tables and analysis
- **Visualizations**: Performance plots and robustness analysis
- **Comprehensive Summary**: Complete results package for download

### Customization Options:
- Modify `CONFIG["models_to_eval"]` to change which models to evaluate
- Adjust audio processing parameters in CONFIG
- Add custom error analysis in the visualization section

---

**🎯 Ready to evaluate ASR robustness on Chinese opera singing!**